## Frengression versus Frugal Flows on the paper's causl binary ATE experiment

This adds Frugal Flows to the **existing binary ATE experiment** (`xwshen51/frengression/paper_exp/binary.ipynb`). The bundled `causl_binary.R` contains its `data.causl` generator and uses the same parameters: 5 instrumental variables, 5 confounders, no outcome-only or spurious covariates, `n=5000`, and a specified ATE of **2**. Both models see the **same observational dataset** per seed. Frengression uses the paper's `num_layer=3`, `hidden_dim=100`, `noise_dim=1`, learning rate `1e-4`, and 1000 updates. The original experiment varies instrument strength over `0, 0.5, 1, 1.5, 2` and repeats 30 times.

The causal margins specified to `causl` are $Y(0)\sim N(0,1)$ and $Y(1)\sim N(2,1)$. This notebook uses their exact ATE, means, and Gaussian laws as ground truth: **it does not draw a separate Monte Carlo reference dataset**. The two fitted models are implicit/sampling models, so their reported ATEs are estimated from finite model draws (`mc=400` per arm); the output includes the associated model-draw standard error. Scalar Wasserstein and energy distances use analytic Gaussian expectations. These are exploratory results until repeated over seeds and instrument strengths.

**Restart the Python kernel, then Run All.** The defaults run all five instrument strengths and 30 seeds, using **400 generated outcomes per arm for both ATE estimates**, as in the original binary notebook. The training sample remains n=5000. Each successful method is checkpointed; a failed fit is recorded and the next seed continues. Run All resumes missing fits after interruption. Old 10,000-draw rows are not reused.

Results are saved under `~/frengression_results/binary_ate/`, including a table with RMSE, Bias and MAE in the existing paper table's order. It is reasonable to add Frugal Flows from this independent replication batch to that table, with its protocol stated; paired comparisons only concern the new Frengression/Frugal Flows fits on shared seeds. Completion counts are saved, so do not report a partial batch as 30 completed repetitions.

R and `causl` must be installed. The setup cell checks dependencies and installs missing Python packages in this kernel. Restart if dependency versions change. The full comparison can run longer than one night; completed fits remain available and are reused.


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

# Works when VS Code uses either the repository root or examples/ as cwd.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "examples" / "compare_frugal_flows_causl.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the frengression_pkg checkout in VS Code and use its notebook kernel.")

# Pinned to a Frugal Flows revision with sample_do and flexible_continuous.
FRUGAL_FLOWS_COMMIT = "26bc25cfa0c262e8a34af61841b202de79ffae0c"
FLOWJAX_VERSION = "19.1.0"  # From frugal-flows/environment.yaml.

def install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *map(str, args)])

# This repository has a src/ layout. Put the checked-out code on the live
# notebook kernel's path even if pip's editable-install metadata is stale.
sys.path.insert(0, str(ROOT / "src"))
if any(importlib.util.find_spec(name) is None for name in ("torch", "engression")):
    install("-e", ROOT)
from frengression import Frengression  # Fail here rather than during the fit.

# Frugal Flows calls fit_to_data(data=...), which older FlowJAX versions do
# not accept. Install the version in Frugal Flows' own environment file.
try:
    installed_flowjax = metadata.version("flowjax")
except metadata.PackageNotFoundError:
    installed_flowjax = None
if installed_flowjax != FLOWJAX_VERSION:
    print(f"FlowJAX {installed_flowjax!r} -> {FLOWJAX_VERSION}; installing compatible version")
    install("flowjax==" + FLOWJAX_VERSION, "lineax==0.1.0", "paramax>=0.0.5")
    importlib.invalidate_caches()

if importlib.util.find_spec("frugal_flows") is None:
    install("git+https://github.com/llaurabatt/frugal-flows.git@" + FRUGAL_FLOWS_COMMIT)
missing = [name for name in ("scipy", "pandas", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    install(*missing)
from frugal_flows.benchmarking import FrugalFlowModel  # Check the API up front.
from flowjax.train import fit_to_data
import inspect
if "data" not in inspect.signature(fit_to_data).parameters:
    raise RuntimeError("Old FlowJAX remains loaded. Restart the notebook kernel and rerun from the first cell.")

sys.path.insert(0, str(ROOT / "examples"))
import importlib
import compare_frugal_flows_causl as comparison
comparison = importlib.reload(comparison)
run, generate_causl, score = comparison.run, comparison.generate_causl, comparison.score
checked_results = comparison.checked_results
import numpy as np
import pandas as pd
from IPython.display import display
import shutil
if shutil.which("Rscript") is None:
    raise RuntimeError("Rscript is missing. Install R and causl; see the first notebook cell.")
check_r = subprocess.run(["Rscript", "-e", "if (!requireNamespace('causl', quietly=TRUE)) quit(status=2)"], capture_output=True, text=True)
if check_r.returncode:
    raise RuntimeError("R package causl is missing. In R, run remotes::install_github('rje42/causl').")
print("Using repository:", ROOT)
print("Python kernel:", sys.executable)


### Confirm the original experiment settings

The following small draw only verifies the R to Python connection and column order. It is **not** used to estimate the causal truth. Observed treatment-arm means are confounded and need not equal 0 and 2.


In [ ]:
x_preview, z_preview, y_preview = generate_causl(n=200, seed=2026, strength_instr=0.0)
print('Covariates:', z_preview.shape[1], '| treatment values:', np.unique(x_preview[:, 0]))
print('Observed arm counts:', {arm: int((x_preview[:, 0] == arm).sum()) for arm in (0, 1)})
print('Exact causal means: do(0)=0, do(1)=2; exact ATE=2')


### Overnight experiment settings

Run All uses 30 seeds at each of five instrument strengths. Both methods use exactly 400 fitted outcome draws per arm for ATE and distribution scores. Frengression retains 1000 updates, noise_dim=1, three layers and width 100; Frugal Flows retains its pinned official 10-D architecture and existing epoch budgets. Optional extra fits are off. Cached rows must include matching draw-count metadata.


In [ ]:
from argparse import Namespace

RUN_PILOT = False          # One-seed pilot already checked.
RUN_GRID_PILOT = False     # Re-run later with the official FF baseline.
RUN_OFFICIAL_FF_CHECK = False  # Already checked; avoid repeating the fit.
GRID_REPEATS = 5
RUN_FR_DIAGNOSTIC = False  # Enable only if the pilot shows underfitting.
RUN_FULL = True             # Five instrument strengths × 30 seeds × two methods.
ATE_DRAWS_PER_ARM = 400
FULL_REPEATS = 30
OUTPUT_DIR = Path.home() / 'frengression_results' / 'binary_ate'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pilot = Namespace(n=5000, repeats=1, seed=2026, strength_instr=0.0,
                  fr_iters=1000, flow_epochs=1000, marginal_epochs=400,
                  mc=ATE_DRAWS_PER_ARM, methods=['frengression', 'frugal_flows_official'],
                  output=str(OUTPUT_DIR / f'causl_pilot_official_mc{ATE_DRAWS_PER_ARM}.csv'))
print('Pilot:', vars(pilot))
print('Frengression: layers=3, hidden_dim=100, noise_dim=1, lr=1e-4')
print('Run pilot:', RUN_PILOT, '| Run five-seed grid:', RUN_GRID_PILOT,
      '| One official FF check:', RUN_OFFICIAL_FF_CHECK,
      '| Fr diagnostic:', RUN_FR_DIAGNOSTIC, '| Full 30 × 5 grid:', RUN_FULL)

assert pilot.mc == 400
print('ATE outcome draws per arm, both methods:', pilot.mc)
print('Results directory:', OUTPUT_DIR)


### One-seed pilot

Check both fitted arm means and signed ATE against their **exact** values. A pilot cannot establish a winner. The ATE standard error in the table reflects finite draws from the fitted model, not uncertainty from training data or across seeds.


In [ ]:
if RUN_PILOT:
    pilot_signature = (pilot.n, pilot.seed, pilot.strength_instr,
                       pilot.fr_iters, pilot.flow_epochs, pilot.marginal_epochs, pilot.mc)
    run(pilot)
    pilot_results = pd.read_csv(pilot.output)
    display(pilot_results[['method', 'n', 'strength_instr', 'true_ate', 'estimated_ate',
                           'ate_error', 'model_draw_ate_se', 'true_mean0', 'estimated_mean0',
                           'true_mean1', 'estimated_mean1', 'wasserstein_mean',
                           'energy_mean', 'quantile_mae', 'seconds']])
else:
    print('Pilot skipped. Set RUN_PILOT=True in the configuration cell.')


### Optional Frengression convergence check

If the pilot underfits, set `RUN_FR_DIAGNOSTIC=True`, rerun the configuration cell, then run this cell. It uses the **same causl data and seed** with 2000 Frengression updates and the unchanged `noise_dim=1` architecture. Inspect arm means and distributional distances; do not tune the model to one favourable seed and then report that seed as final.


In [ ]:
if RUN_FR_DIAGNOSTIC:
    current_signature = (pilot.n, pilot.seed, pilot.strength_instr,
                         pilot.fr_iters, pilot.flow_epochs, pilot.marginal_epochs, pilot.mc)
    if 'pilot_signature' not in globals() or pilot_signature != current_signature:
        raise RuntimeError('Run the pilot again after changing its configuration.')
    settings = Namespace(**vars(pilot))
    settings.fr_iters = 2000
    settings.methods = ['frengression']
    settings.output = str(OUTPUT_DIR / 'causl_fr_2000.csv')
    run(settings)
    diagnostic = pd.concat([pilot_results.loc[pilot_results.method == 'frengression'],
                            pd.read_csv(settings.output)], ignore_index=True)
    display(diagnostic[['fr_iters', 'estimated_ate', 'ate_error', 'model_draw_ate_se',
                        'estimated_mean0', 'estimated_mean1', 'wasserstein_mean',
                        'energy_mean', 'seconds']])
else:
    print('Frengression diagnostic skipped.')


### Five-seed grid with the current Frugal Flows configuration

The previous five-seed results were produced with a compact FF architecture and remain saved under their own filenames. This cell is now **off** by default. When `RUN_GRID_PILOT=True`, it will use the authors' wider 10-D benchmarking configuration through `frugal_flows_official`, with `flow_epochs=1000`; its filenames include that new epoch limit, so it will not overwrite the old results. Each completed pair is saved separately and can be reused after interruption. This can take substantially longer than the compact run.


In [ ]:
if RUN_GRID_PILOT:
    tag = (f'n{pilot.n}_fr{pilot.fr_iters}_ff{pilot.flow_epochs}_'
           f'm{pilot.marginal_epochs}_mc{pilot.mc}')
    combined_path = OUTPUT_DIR / f'causl_grid5_{tag}.csv'
    collected = []
    for strength in np.arange(0, 2.5, 0.5):
        for rep in range(GRID_REPEATS):
            settings = Namespace(**vars(pilot))
            settings.seed = pilot.seed + rep
            settings.repeats = 1
            settings.strength_instr = float(strength)
            settings.output = str(
                OUTPUT_DIR / f'causl_grid5_{tag}_s{strength:.1f}_seed{settings.seed}.csv')
            result_file = Path(settings.output)
            if result_file.is_file():
                rows = pd.read_csv(result_file)
                if (len(rows) != 2
                    or set(rows.method) != set(settings.methods)
                    or not (rows.seed == settings.seed).all()
                    or not (rows.n == settings.n).all()
                    or not np.allclose(rows.strength_instr, strength)
                    or not (rows.true_ate == 2.0).all()):
                    raise RuntimeError(f'Unexpected cached result: {result_file}')
                print('Using completed pair:', result_file.name)
            else:
                print(f'Running strength={strength:.1f}, seed={settings.seed}')
                run(settings)
                rows = pd.read_csv(result_file)
            collected.append(rows)
            pd.concat(collected, ignore_index=True).to_csv(combined_path, index=False)
    grid_results = pd.concat(collected, ignore_index=True)
    grid_results['signed_error'] = grid_results.estimated_ate - grid_results.true_ate
    grid_summary = grid_results.groupby(['strength_instr', 'method']).agg(
        seeds=('seed', 'nunique'),
        ate_bias=('signed_error', 'mean'),
        ate_mae=('ate_error', 'mean'),
        ate_rmse=('signed_error', lambda v: float(np.sqrt(np.mean(np.square(v))))),
        wasserstein=('wasserstein_mean', 'mean'),
        energy=('energy_mean', 'mean'),
        quantile_mae=('quantile_mae', 'mean')
    )
    display(grid_summary.round(4))
    print('Combined results:', combined_path)
else:
    print('Five-seed grid skipped. Set RUN_GRID_PILOT=True and rerun the configuration cell.')


### Optional one-seed check of the official Frugal Flows configuration

With `RUN_OFFICIAL_FF_CHECK=True`, this cell fits only `frugal_flows_official` at strength 2, seed 2026, n=5000. It imports the architecture and learning rate from the pinned Frugal Flows `benchmarking.py`, with 1000 flow epochs and 400 marginal-CDF epochs. It uses the configured **400 outcome draws per arm**. One fit is a diagnostic; the default full grid is the repeated comparison. Earlier compact 10,000-draw results are not mixed with this check.


In [ ]:
if RUN_OFFICIAL_FF_CHECK:
    from frugal_flows.benchmarking import hyperparam_dict, causal_margin_hyperparam_dict
    print('Pinned FF 10-D copula settings:', hyperparam_dict)
    print('Pinned FF causal margin settings:', causal_margin_hyperparam_dict)
    official = Namespace(**vars(pilot))
    official.strength_instr = 2.0
    official.seed = 2026
    official.repeats = 1
    official.methods = ['frugal_flows_official']
    official.output = str(OUTPUT_DIR / (
        f'causl_ffofficial_n{official.n}_ep{official.flow_epochs}_'
        f'm{official.marginal_epochs}_mc{official.mc}_s2.0_seed{official.seed}.csv'))
    run(official)
    official_path = Path(official.output)
    if official_path.is_file():
        official_result = checked_results(pd.read_csv(official_path), official)
        display(official_result[['method', 'seed', 'strength_instr', 'mc_draws_per_arm',
            'draws0', 'draws1', 'estimated_ate', 'ate_error', 'model_draw_ate_se',
            'wasserstein_mean', 'energy_mean', 'quantile_mae', 'seconds']])
    else:
        print('No completed fit; inspect its failure log in', OUTPUT_DIR)
else:
    print('Official Frugal Flows check skipped.')


### Full grid, saved progress and paper table

This cell attempts 30 seeds per strength. The helper saves each successful method immediately and resumes only missing fits; failures remain in the logs. Summaries show completed counts. Paired performance differences use complete shared-seed pairs only. Bootstrap intervals describe variation of the **simulation performance comparison**, not confidence intervals for a causal effect on one dataset. The `mc400` output filenames and cache checks prevent reusing old 10,000-draw outcomes.


In [ ]:
if RUN_FULL:
    strengths = [float(v) for v in np.arange(0, 2.5, 0.5)]
    full_repeats = FULL_REPEATS
    tag = (f'n{pilot.n}_fr{pilot.fr_iters}_ff{pilot.flow_epochs}_'
           f'm{pilot.marginal_epochs}_mc{pilot.mc}')
    full_path = OUTPUT_DIR / f'causl_full30_{tag}.csv'
    collected, attempt_failures, completion_rows = [], [], []
    for strength in strengths:
        for rep in range(full_repeats):
            settings = Namespace(**vars(pilot))
            settings.seed = pilot.seed + rep
            settings.repeats = 1
            settings.strength_instr = strength
            pair_path = OUTPUT_DIR / f'causl_full30_{tag}_s{strength:.1f}_seed{settings.seed}.csv'
            settings.output = str(pair_path)
            print(f'Attempting strength={strength:.1f}, seed={settings.seed}', flush=True)
            try:
                run(settings)
                rows = checked_results(pd.read_csv(pair_path), settings)
            except Exception as error:
                attempt_failures.append(dict(strength_instr=strength, seed=settings.seed,
                                             method='checkpoint_or_run', error=str(error)))
                rows = pd.DataFrame(columns=comparison.RESULT_COLUMNS)
                print(f'Could not complete this seed: {error}', flush=True)
            if not rows.empty:
                collected.append(rows)
            completed_methods = set(rows.method)
            completion_rows.append(dict(strength_instr=strength, seed=settings.seed,
                frengression_complete='frengression' in completed_methods,
                frugal_flows_complete='frugal_flows_official' in completed_methods,
                complete_pair=set(settings.methods).issubset(completed_methods)))
            failure_path = pair_path.with_name(pair_path.stem + '_failures.csv')
            if failure_path.is_file():
                failures = pd.read_csv(failure_path)
                for record in failures.to_dict('records'):
                    attempt_failures.append({'strength_instr': strength, **record})
            if collected:
                pd.concat(collected, ignore_index=True).to_csv(full_path, index=False)
            pd.DataFrame(completion_rows).to_csv(OUTPUT_DIR / f'completion_{tag}.csv', index=False)
            pd.DataFrame(attempt_failures, columns=['strength_instr', 'seed', 'method', 'stage',
                'error', 'attempted_at_utc']).to_csv(OUTPUT_DIR / f'failures_{tag}.csv', index=False)
    if not collected:
        raise RuntimeError(f'No completed fits. Inspect failure files in {OUTPUT_DIR}')
    full_results = pd.concat(collected, ignore_index=True)
    full_results['signed_error'] = full_results.estimated_ate - full_results.true_ate
    assert (full_results.mc_draws_per_arm == 400).all()
    assert (full_results.draws0 == 400).all() and (full_results.draws1 == 400).all()
    full_summary = full_results.groupby(['strength_instr', 'method']).agg(
        seeds=('seed', 'nunique'), mc_draws_per_arm=('mc_draws_per_arm', 'first'),
        ate_bias=('signed_error', 'mean'), ate_mae=('ate_error', 'mean'),
        ate_rmse=('signed_error', lambda v: float(np.sqrt(np.mean(np.square(v))))),
        mean_model_draw_ate_se=('model_draw_ate_se', 'mean'),
        wasserstein=('wasserstein_mean', 'mean'), energy=('energy_mean', 'mean'),
        quantile_mae=('quantile_mae', 'mean'))
    summary_path = OUTPUT_DIR / f'causl_full30_summary_{tag}.csv'
    full_summary.to_csv(summary_path)
    display(full_summary.round(4))
    table_rows = []
    for (strength, method), row in full_summary.iterrows():
        for label, field in [('RMSE', 'ate_rmse'), ('Bias', 'ate_bias'), ('MAE', 'ate_mae')]:
            table_rows.append(dict(strength_instr=strength, metric=label, method=method, value=row[field]))
    paper_ate_table = pd.DataFrame(table_rows).pivot(
        index=['strength_instr', 'metric'], columns='method', values='value')
    paper_ate_table = paper_ate_table.reindex(pd.MultiIndex.from_product(
        [strengths, ['RMSE', 'Bias', 'MAE']], names=['strength_instr', 'metric']))
    paper_table_path = OUTPUT_DIR / f'binary_ate_table_{tag}.csv'
    paper_ate_table.to_csv(paper_table_path)
    print('ATE table: existing paper metric order; inspect completion counts before use')
    display(paper_ate_table.round(3))

    metrics = ['ate_error', 'wasserstein_mean', 'energy_mean', 'quantile_mae']
    paired_rows = []
    for strength in strengths:
        block = full_results.loc[full_results.strength_instr == strength]
        if not {'frengression', 'frugal_flows_official'}.issubset(set(block.method)):
            continue
        wide = block.pivot(index='seed', columns='method', values=metrics).dropna()
        count = len(wide)
        if count == 0:
            continue
        rng = np.random.default_rng(2026 + int(10 * strength))
        indices = rng.integers(0, count, size=(2000, count))
        for metric in metrics:
            delta = (wide[metric, 'frengression'] - wide[metric, 'frugal_flows_official']).to_numpy()
            bounds = np.quantile(delta[indices].mean(axis=1), [0.025, 0.975]) if count > 1 else [np.nan, np.nan]
            paired_rows.append(dict(strength_instr=strength, metric=metric, paired_seeds=count,
                mean_fr_minus_ff=float(delta.mean()), ci95_low=float(bounds[0]),
                ci95_high=float(bounds[1]), fr_better_seeds=int((delta < 0).sum())))
    paired_summary = pd.DataFrame(paired_rows)
    paired_path = OUTPUT_DIR / f'causl_full30_paired_{tag}.csv'
    paired_summary.to_csv(paired_path, index=False)
    completion = pd.DataFrame(completion_rows)
    print('Completed pairs per instrument strength (planned 30):')
    display(completion.groupby('strength_instr').complete_pair.sum())
    print('Paired comparison: Frengression error minus Frugal Flows error; negative favors Frengression.')
    display(paired_summary.round(4))
    print('Results directory:', OUTPUT_DIR)
    print('Paper table:', paper_table_path)
else:
    print('Full grid skipped. Set RUN_FULL=True in the configuration cell.')


### Interpretation and reproducibility

These are paired comparisons in the shared **binary-treatment, scalar continuous-outcome** setting. Negative values of `mean_fr_minus_ff` favour Frengression; the 95% intervals resample matched seeds separately within each strength. The Frugal Flows architecture and learning rate come from its authors' pinned 10-D benchmark, with the maximum epochs set here to 1000 (flow) and 400 (marginal CDF). The scalar `energy_mean` uses SciPy's square-root convention and is not the squared energy distance in the theory. Report the exact configuration, all strengths, and both methods; five-seed pilot results alone are not a final comparison. Joint outcomes and longitudinal data need separate experiments.


In [ ]:
for package in ('frengression', 'frugal-flows', 'torch', 'jax', 'flowjax', 'numpy', 'scipy'):
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: version unavailable')
try:
    print('frengression_pkg commit:', subprocess.check_output(
        ['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print('Frugal Flows install target:', FRUGAL_FLOWS_COMMIT)
